# Qdrant + Ollama RAG Flow Playground
This notebook demonstrates the end-to-end RAG cycle:
1. Searching Qdrant database to retrieve relevant context vectors
2. Formatting a context-aware system prompt
3. Generating final assistant completion from the configured LLM reasoning model.

In [1]:
import sys
from pathlib import Path

# Locate and append project root
def get_project_root() -> Path:
    current = Path.cwd()
    for _ in range(5):
        if (current / "pyproject.toml").exists():
            return current
        current = current.parent
    return Path.cwd()

sys.path.append(str(get_project_root().resolve()))

In [2]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct, Filter, FieldCondition, MatchValue
import httpx
import uuid
from src.ragforge.config import QDRANT_URL, OLLAMA_URL, DEFAULT_EMBEDDING_MODEL, DEFAULT_LLM_MODEL, DEFAULT_COLLECTION

client = QdrantClient(url=QDRANT_URL)
test_collection = "notebook-rag-collection"

# 1. Clean up & Create Collection
if client.collection_exists(test_collection):
    client.delete_collection(test_collection)

client.create_collection(
    collection_name=test_collection,
    vectors_config=VectorParams(size=768, distance=Distance.COSINE)
)

True

## 2. Seed Mock Document Knowledge Base

In [3]:
def get_embedding(text: str) -> list[float]:
    res = httpx.post(
        f"{OLLAMA_URL}/api/embeddings",
        json={"model": DEFAULT_EMBEDDING_MODEL, "prompt": text},
        timeout=15.0
    )
    return res.json()["embedding"]

kb_docs = [
    "Temporal workers poll task queues and execute workflows sequentially.",
    "Qdrant offers payload index optimization for rapid filtering.",
    "OpenProject enables Agile boards and Scrum backlog tracking."
]

points = []
for doc in kb_docs:
    points.append(PointStruct(
        id=str(uuid.uuid4()),
        vector=get_embedding(doc),
        payload={"text": doc}
    ))

client.upsert(collection_name=test_collection, points=points)
print("Seeded knowledge base!")

Seeded knowledge base!


## 3. Retrieve Context & Query LLM (RAG)

In [4]:
user_query = "How does project management work in OpenProject?"
query_vector = get_embedding(user_query)

# Search vector DB
hits = client.query_points(
    collection_name=test_collection,
    query=query_vector,
    limit=1
).points

context = ""
if hits:
    context = hits[0].payload["text"]
    print(f"Retrieved Context: '{context}'")
else:
    print("No relevant context found.")

# Construct RAG prompt
system_prompt = f"You are a helpful project assistant. Use the following context to answer the question:\nContext: {context}"

# Call LLM
response = httpx.post(
    f"{OLLAMA_URL}/api/chat",
    json={
        "model": DEFAULT_LLM_MODEL,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_query}
        ],
        "stream": False
    },
    timeout=45.0
)
response.raise_for_status()
answer = response.json()["message"]["content"]
print(f"\nAssistant Answer:\n{answer}")

Retrieved Context: 'OpenProject enables Agile boards and Scrum backlog tracking.'

Assistant Answer:
Based on the context provided, OpenProject helps with project management by enabling specific methodologies and tools:

*   **Agile Boards:** It provides functionality for Agile boards, allowing teams to visualize their workflow and manage tasks iteratively.
*   **Scrum Backlog Tracking:** It specifically supports Scrum backlog tracking, which is crucial for managing product requirements and planning sprints within the Scrum framework.

In essence, OpenProject equips users with tools tailored for both Agile and Scrum project management styles.


## 4. Clean up

In [5]:
client.delete_collection(test_collection)
print("Cleaned up temporary collection successfully.")

Cleaned up temporary collection successfully.
